In [1]:
import zipfile
import os
import pandas as pd

with zipfile.ZipFile("CA-2023.zip", "r") as zip_ref:
    zip_ref.extractall("CA_data/CA-2023/CA")
os.listdir("CA_data/CA-2023/CA")

['agencies.csv',
 'CA-2023',
 'NIBRS_ACTIVITY_TYPE.csv',
 'NIBRS_AGE.csv',
 'NIBRS_ARRESTEE.csv',
 'NIBRS_ARRESTEE_GROUPB.csv',
 'NIBRS_ARRESTEE_GROUPB_WEAPON.csv',
 'NIBRS_ARRESTEE_WEAPON.csv',
 'NIBRS_ARREST_TYPE.csv',
 'NIBRS_ASSIGNMENT_TYPE.csv',
 'NIBRS_BIAS_LIST.csv',
 'NIBRS_BIAS_MOTIVATION.csv',
 'NIBRS_CIRCUMSTANCES.csv',
 'NIBRS_CLEARED_EXCEPT.csv',
 'NIBRS_CRIMINAL_ACT.csv',
 'NIBRS_CRIMINAL_ACT_TYPE.csv',
 'nibrs_diagram.pdf',
 'NIBRS_DRUG_MEASURE_TYPE.csv',
 'NIBRS_ETHNICITY.csv',
 'NIBRS_incident.csv',
 'NIBRS_INJURY.csv',
 'NIBRS_JUSTIFIABLE_FORCE.csv',
 'NIBRS_LOCATION_TYPE.csv',
 'NIBRS_month.csv',
 'NIBRS_OFFENDER.csv',
 'NIBRS_OFFENSE.csv',
 'NIBRS_OFFENSE_TYPE.csv',
 'NIBRS_PROPERTY.csv',
 'NIBRS_PROPERTY_DESC.csv',
 'NIBRS_PROP_DESC_TYPE.csv',
 'NIBRS_PROP_LOSS_TYPE.csv',
 'NIBRS_RELATIONSHIP.csv',
 'NIBRS_SUSPECTED_DRUG.csv',
 'NIBRS_SUSPECTED_DRUG_TYPE.csv',
 'NIBRS_SUSPECT_USING.csv',
 'NIBRS_USING_LIST.csv',
 'NIBRS_VICTIM.csv',
 'NIBRS_VICTIM_CIRCUMSTANCES.csv

In [2]:

month = pd.read_csv("CA_data/CA-2023/CA/NIBRS_month.csv", usecols=["nibrs_month_id","agency_id","data_year"])

agencies = pd.read_csv("CA_data/CA-2023/CA/agencies.csv", usecols=["agency_id","county_name"], encoding="latin1")

month_agency = month.merge(agencies, on="agency_id")
month_agency = month_agency[month_agency["data_year"] == 2023]

In [3]:
#mapping from nibrs_month_id to county_name
month_to_county = month_agency.set_index("nibrs_month_id")["county_name"].to_dict()

#incident counts by county
counts = {}

for chunk in pd.read_csv("CA_data/CA-2023/CA/NIBRS_incident.csv", usecols=["nibrs_month_id", "incident_id"], chunksize=200000):
    #add county_name to each row 
    chunk['county_name'] = chunk['nibrs_month_id'].map(month_to_county)
    
    #remove rows where it can't be mapped
    chunk = chunk.dropna(subset=['county_name'])
    
    #remove duplicate incident_id within the same county
    unique_incidents = chunk.drop_duplicates(subset=['incident_id', 'county_name'])
    
    #count unique incidents by county
    county_counts = unique_incidents['county_name'].value_counts()
    
    #add the counts from this chunk to the running total
    for county, count in county_counts.items():
        counts[county] = counts.get(county, 0) + count

print("\nFinal unique incident counts by county:")
for county, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{county}: {count:,}")


Final unique incident counts by county:
LOS ANGELES: 137,529
SAN DIEGO: 115,222
ORANGE: 108,922
SACRAMENTO: 70,057
SANTA CLARA: 61,246
ALAMEDA: 52,408
CONTRA COSTA: 48,431
RIVERSIDE: 46,881
FRESNO: 43,599
SAN JOAQUIN: 34,796
KERN: 33,155
SAN MATEO: 29,588
SAN BERNARDINO: 28,670
TULARE: 26,203
SOLANO: 22,310
STANISLAUS: 20,115
VENTURA: 19,204
SONOMA: 17,033
MERCED: 14,733
MONTEREY: 14,313
SANTA BARBARA: 13,344
PLACER: 10,072
BUTTE: 9,742
YOLO: 8,023
SAN LUIS OBISPO: 7,920
HUMBOLDT: 7,065
MARIN: 7,027
MADERA: 6,700
SANTA CRUZ: 5,929
EL DORADO: 5,722
KINGS: 5,541
NAPA: 5,462
IMPERIAL: 5,109
SHASTA: 5,071
SUTTER: 4,206
TEHAMA: 3,845
LAKE: 3,447
NEVADA: 2,682
TUOLUMNE: 2,084
SAN BENITO: 1,902
MENDOCINO: 1,762
AMADOR: 1,564
CALAVERAS: 1,533
LASSEN: 1,084
YUBA: 1,036
SISKIYOU: 1,029
DEL NORTE: 992
INYO: 923
MARIPOSA: 861
COLUSA: 833
SAN FRANCISCO: 815
NOT SPECIFIED: 763
TRINITY: 578
GLENN: 488
MONO: 486
MODOC: 274
ALPINE: 46
SIERRA: 46
PLUMAS: 9


In [4]:
import pandas as pd
import os

#crime counts
crime_counts_df = pd.DataFrame(list(counts.items()), columns=['Area_Name_raw', 'incident_count'])

#socioeconomic data
combined_data = pd.read_csv('CombinedData.csv')

#unemployment data has different format
unemployment_data = combined_data[combined_data['Dataset'] == 'unemployment'].copy()
other_data = combined_data[combined_data['Dataset'] != 'unemployment'].copy()

#pivot the non-unemployment data
pivot_other = other_data.pivot_table(index=['FIPS_Code', 'State', 'Area_Name'], columns='Attribute', values='Value', aggfunc='first').reset_index()
pivot_other.columns.name = None

#handle unemployment data separately
unemployment_data['County'] = unemployment_data['Area_Name'].str.replace(', CA', '', regex=False)

# Pivot unemployment data
pivot_unemployment = unemployment_data.pivot_table(
    index=['FIPS_Code', 'State', 'County'],
    columns='Attribute',
    values='Value',
    aggfunc='first'
).reset_index()
pivot_unemployment.columns.name = None

# Standardize county names for crime data
crime_counts_df['County'] = crime_counts_df['Area_Name_raw'].str.title() + ' County'

# Create a clean county column in pivot_other
pivot_other['County'] = pivot_other['Area_Name'].str.replace(', CA', '', regex=False)

# Drop the original Area_Name column
pivot_other = pivot_other.drop(columns=['Area_Name'])

# Remove duplicates
pivot_other = pivot_other.drop_duplicates(subset=['County'])

#merge with crime data
merged_df = pd.merge(pivot_other, crime_counts_df[['County', 'incident_count']], on='County',how='inner')

#merge with unemployment data
merged_df = pd.merge(merged_df,
    pivot_unemployment[['County', 'Civilian_labor_force_2023', 'Employed_2023', 'Unemployed_2023', 'Unemployment_rate_2023']],
    on='County',
    how='left')

cols = merged_df.columns.tolist()

#move County to the front
if 'County' in cols:
    cols.remove('County')
    cols.insert(0, 'County')

#move incident_count
if 'incident_count' in cols:
    cols.remove('incident_count')
    cols.insert(1, 'incident_count')

#reorder the dataframe
merged_df = merged_df[cols]

merged_df.to_csv('CA_county_crime_FINAL.csv', index=False)